# Theseus 教程（中文翻译版）

- 原始英文版：`04_motion_planning.ipynb`
- 说明：本文件为自动翻译版本（保留代码不翻译，清空输出以减小体积）。如遇术语不一致，可优先参考英文原文。


# 运动规划第 1 部分：作为非线性最小二乘优化的运动规划

在本教程中，我们将学习如何为平面环境中的 2D 机器人实现 [GPMP2](https://journals.sagepub.com/doi/pdf/10.1177/0278364918790369)（Mukadam 等人，2018）运动规划算法。

目标是在给定起始和目标姿势以及环境的某些表示的情况下找到机器人的轨迹（姿势和速度）。这可以作为优化问题来解决，其中要优化的变量是机器人沿着一些总时间步长（以某个固定时间间隔）的轨迹的 2D 位姿和 2D 速度。在此示例中，我们使用以下成本项制定优化目标，并通过各自的权重进行平衡：
* **边界条件**：轨迹应以零速度开始于起始姿势，以零速度结束于目标姿势。
* **避免碰撞**：轨迹应避免与环境中的障碍物发生碰撞（我们使用带符号的距离场）。
* **平滑度**：轨迹应该是平滑的（我们使用零加速度先验）。

In [ ]:
import random

import matplotlib as mpl
import numpy as np
import torch
import torch.utils.data

import theseus as th
import theseus.utils.examples as theg

%load_ext autoreload
%autoreload 2

torch.set_default_dtype(torch.double)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
seed = 0
torch.random.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["font.size"] = 16

## 1. 加载并可视化轨迹数据

首先，让我们从使用 [dgpmp2](https://github.com/mhmukadam/dgpmp2) 中的代码生成的地图和轨迹数据集中加载一些运动规划问题。

In [ ]:
dataset_dir = "data/motion_planning_2d"
dataset = theg.TrajectoryDataset(True, 2, dataset_dir, map_type="tarpit")
data_loader = torch.utils.data.DataLoader(dataset, 2)

batch = next(iter(data_loader))

该批次是 `torch.Tensor` 的字符串字典，包含以下键：

In [ ]:
for k, v in batch.items():
    if k != "file_id":
        print(f"{k:20s}: {v.shape}")

让我们绘制加载的地图和轨迹。 `th.eb.SignedDistanceField2D` 是一个带符号的距离场对象，其中包括一个将 *x,y* 坐标转换为我们在此处用于绘图的地图单元格的函数。为了完整起见，我们显示了加载的专家轨迹，尽管我们不会在本示例中使用它们（我们将在本教程的第 2 部分中这样做）。我们还说明了每张地图的带符号距离场。

In [ ]:
sdf = th.eb.SignedDistanceField2D(
    th.Point2(batch["sdf_origin"]),
    th.Variable(batch["cell_size"]),
    th.Variable(batch["sdf_data"]),
)
figs = theg.generate_trajectory_figs(
    batch["map_tensor"], 
    sdf, 
    [batch["expert_trajectory"]], 
    robot_radius=0.4, 
    labels=["expert trajectory"], 
    fig_idx_robot=0,
    figsize=(10, 4),
    plot_sdf=True,
)
figs[0].show()
figs[1].show()

以下是我们将在整个示例中使用的一些常量

In [ ]:
trajectory_len = batch["expert_trajectory"].shape[2]
num_time_steps = trajectory_len - 1
map_size = batch["map_tensor"].shape[1]
safety_distance = 0.4
robot_radius = 0.4
total_time = 10.0
dt_val = total_time / num_time_steps
Qc_inv = [[1.0, 0.0], [0.0, 1.0]]
collision_w = 20.0
boundary_w = 100.0

## 2. 问题建模

### 2.1。定义变量对象

我们在此示例中的目标是使用 `Theseus` 为上面加载的地图生成计划。正如简介中提到的，我们需要沿着要优化的轨迹的每个点的 2D 位姿和 2D 速度。为此，我们将创建一组具有单独名称的 `th.Point2` 变量，并将它们存储在两个列表中，以便稍后可以将它们传递给适当的成本函数。

In [ ]:
# Create optimization variables
poses = []
velocities = []
for i in range(trajectory_len):
    poses.append(th.Point2(name=f"pose_{i}", dtype=torch.double))
    velocities.append(th.Point2(name=f"vel_{i}", dtype=torch.double))

除了优化变量之外，我们还需要一组“辅助”变量来包装成本函数计算中涉及的与地图相关的量，但这些变量在整个优化过程中都是恒定的。这包括开始/目标目标值，以及碰撞和动态成本函数的参数。

In [ ]:
# Targets for pose boundary cost functions
start_point = th.Point2(name="start")
goal_point = th.Point2(name="goal")

# For collision avoidance cost function
sdf_origin = th.Point2(name="sdf_origin")
cell_size = th.Variable(torch.empty(1, 1), name="cell_size")
sdf_data = th.Variable(torch.empty(1, map_size, map_size), name="sdf_data")
cost_eps = th.Variable(torch.tensor(robot_radius + safety_distance).view(1, 1), name="cost_eps")

# For GP dynamics cost function
dt = th.Variable(torch.tensor(dt_val).view(1, 1), name="dt")

### 2.2。成本权重

接下来，我们将创建一系列成本权重，用于优化问题中涉及的每个成本函数。

In [ ]:
# Cost weight to use for all GP-dynamics cost functions
gp_cost_weight = th.eb.GPCostWeight(torch.tensor(Qc_inv), dt)

# Cost weight to use for all collision-avoidance cost functions
collision_cost_weight = th.ScaleCostWeight(th.Variable(torch.tensor(collision_w)))

# For all hard-constraints (end points pos/vel) we use a single scalar weight
# with high value
boundary_cost_weight = th.ScaleCostWeight(boundary_w)

### 2.3。成本函数

在本节中，我们现在将创建一个 `Theseus` 目标并添加用于运动规划的 GPMP2 成本函数。首先，我们制定目标：

In [ ]:
objective = th.Objective(dtype=torch.double)

#### 边界成本函数

这里我们为边界条件创建成本函数，为其分配名称，并将它们添加到 `Objective`。对于边界，我们需要四个成本函数，并且对于每个我们使用 `th.Difference` 类型的成本函数。该成本函数类型将优化变量、成本权重、目标辅助变量和名称作为输入。其误差函数是优化变量与目标之间的局部差异。

例如，考虑下面添加的第一个 `Difference`（名称为 `pose_0`）。该成本函数将告诉优化器尝试使 `poses[0]` 处的变量值接近辅助变量 `start_point`（这也是一个命名变量，如第 2.1 节中所述）的值。另一方面，对于速度约束（对于 `vel_0`），我们不需要为目标传递*命名*辅助变量，因为我们知道我们希望它是 `torch.zeros(1, 2)`，无论地图数据是什么（机器人以零速度开始）。

最后，所有这些成本函数共享相同的boundary_cost_weight，您可能还记得，它是`ScaleCostWeight(100.0)`。

In [ ]:
# Fixed starting position
objective.add(
    th.Difference(poses[0], start_point, boundary_cost_weight, name="pose_0")
)
# Fixed initial velocity
objective.add(
    th.Difference(
        velocities[0],
        th.Point2(tensor=torch.zeros(1, 2)),
        boundary_cost_weight,
        name="vel_0",
    )
)
objective.add(
    th.Difference(
        poses[-1], goal_point, boundary_cost_weight, name="pose_N"
    )
)
objective.add(
    th.Difference(
        velocities[-1],
        th.Point2(tensor=torch.zeros(1, 2)),
        boundary_cost_weight,
        name="vel_N",
    )
)

#### 碰撞成本函数

为了避免冲突，我们使用 `th.eb.Collision2D` 成本函数类型，它接收以下输入：

* 单个 `th.Point2` 优化变量。
* 辅助变量：
    * 三个代表有符号距离场数据（sdf_origin、sdf_data、cell_size）。
    * 产生碰撞成本的距离 (cost_eps)。
* 成本权重。
    
由于轨迹中的每个内部点都需要一个这样的成本函数，因此我们在循环中创建成本函数并传递上面定义的相应位姿变量。

In [ ]:
for i in range(1, trajectory_len - 1):
    objective.add(
        th.eb.Collision2D(
            poses[i],
            sdf_origin,
            sdf_data,
            cell_size,
            cost_eps,
            collision_cost_weight,
            name=f"collision_{i}",
        )
    )

#### GP 动力学成本函数

为了确保平滑的轨迹，我们使用 `th.eb.GPMotionModel` 成本函数，它接收以下输入：
 
* 四个 `th.Point2` 优化变量：时间 t-1 时的位姿、时间 t-1 时的速度、时间 t 时的位姿、时间 t 时的速度。
* 一个辅助变量描述连续时间步之间的时间差。
* 成本权重（通常为 `th.eb.GPCostWeight` 类型）。

对于每一对连续状态（位姿和速度），我们需要一个这样的成本函数，因此我们将它们添加到循环中，并从上面创建的列表中传递适当的优化变量。

In [ ]:
for i in range(1, trajectory_len):
    objective.add(
        (
            th.eb.GPMotionModel(
                poses[i - 1],
                velocities[i - 1],
                poses[i],
                velocities[i],
                dt,
                gp_cost_weight,
                name=f"gp_{i}",
            )
        )
    )

## 创建用于运动规划的TheseusLayer

在本例中，我们将使用 Levenberg-Marquardt 作为非线性优化器，并结合基于 Cholesky 分解的密集线性求解器。

In [ ]:
optimizer = th.LevenbergMarquardt(
    objective,
    th.CholeskyDenseSolver,
    max_iterations=50,
    step_size=1.0,
)
motion_planner = th.TheseusLayer(optimizer)
motion_planner.to(device=device, dtype=torch.double)

## 3. 运行优化器

最后，我们准备生成一些最佳计划。我们首先初始化所有辅助变量，其值与地图相关（例如，起始位置和目标位置，或 SDF 数据）。我们还为优化变量提供了一些合理的初始值；在此示例中，我们将初始化优化变量，使之从开始到目标位于一条直线上。以下辅助函数对此很有用：

In [ ]:
def get_straight_line_inputs(start, goal):
    # Returns a dictionary with pose and velocity variable names associated to a 
    # straight line trajectory between start and goal
    start_goal_dist = goal - start
    avg_vel = start_goal_dist / total_time
    unit_trajectory_len = start_goal_dist / (trajectory_len - 1)
    input_dict = {}
    for i in range(trajectory_len):
        input_dict[f"pose_{i}"] = start + unit_trajectory_len * i
        if i == 0 or i == trajectory_len - 1:
            input_dict[f"vel_{i}"] = torch.zeros_like(avg_vel)
        else:
            input_dict[f"vel_{i}"] = avg_vel
    return input_dict

现在，让我们将运动规划数据传递给 `TheseusLayer` 并开始创建一些轨迹；请注意，我们可以利用Theseus 的批量支持同时求解两个轨迹。为了初始化变量，我们创建一个将字符串映射到 `torch.Tensor` 的字典，其中键是 `th.Variable` 名称，值是应用于其初始值的张量。

In [ ]:
start = batch["expert_trajectory"][:, :2, 0].to(device)
goal = batch["expert_trajectory"][:, :2, -1].to(device)
planner_inputs = {
    "sdf_origin": batch["sdf_origin"].to(device),
    "start": start.to(device),
    "goal": goal.to(device),
    "cell_size": batch["cell_size"].to(device),
    "sdf_data": batch["sdf_data"].to(device),
}
planner_inputs.update(get_straight_line_inputs(start, goal))    
with torch.no_grad():        
    final_values, info = motion_planner.forward(
        planner_inputs,
        optimizer_kwargs={
            "track_best_solution": True,
            "verbose": True,
            "damping": 0.1,
        }
    )

## 4. 结果

优化完成后，我们可以查询优化变量以获得最终的轨迹并将结果可视化。以下函数从 `TheseusLayer` 的输出字典创建轨迹张量。

In [ ]:
def get_trajectory(values_dict):
    trajectory = torch.empty(values_dict[f"pose_0"].shape[0], 4, trajectory_len, device=device)
    for i in range(trajectory_len):
        trajectory[:, :2, i] = values_dict[f"pose_{i}"]
        trajectory[:, 2:, i] = values_dict[f"vel_{i}"]
    return trajectory

现在让我们绘制最终的轨迹

In [ ]:
trajectory = get_trajectory(info.best_solution).cpu()

sdf = th.eb.SignedDistanceField2D(
    th.Point2(batch["sdf_origin"]),
    th.Variable(batch["cell_size"]),
    th.Variable(batch["sdf_data"]),
)
figs = theg.generate_trajectory_figs(
    batch["map_tensor"], 
    sdf, 
    [trajectory], 
    robot_radius=robot_radius, 
    labels=["solution trajectory"], 
    fig_idx_robot=0,
    figsize=(6, 6)
)
figs[0].show()
figs[1].show()